# Data Aggregation

In [0]:
# Create schema to RetailCast data
spark.sql("""
CREATE SCHEMA IF NOT EXISTS retailcast_solo.retailcast_solo_gold
""")

DataFrame[]

In [0]:
%sql
-- ======================================================
-- Para cada versão, aumentar o número incremental da versão, para registo do histórico e rastreabilidade
-- ======================================================

CREATE OR REPLACE TABLE retailcast_solo.retailcast_solo_gold.rc_aggregation_gold AS

WITH Eventos_Expandidos AS (
    -- Expande os intervalos de datas dos eventos para criar um registo por dia
    SELECT
        FK_LOJA,
        EXPLODE(SEQUENCE(TO_DATE(DATA_INI), TO_DATE(DATA_FIM), INTERVAL 1 DAY)) AS DATA,
        DESCRICAO AS EVENTO_DESCRICAO
    FROM retailcast_solo.retailcast_solo_silver.rc_eventos_silver
),
Eventos_Diarios AS (
    -- Agrupa múltiplos eventos simultâneos para evitar duplicação dos valores da target no JOIN
    SELECT
        FK_LOJA,
        DATA,
        CONCAT_WS(' | ', COLLECT_SET(EVENTO_DESCRICAO)) AS EVENTOS
    FROM Eventos_Expandidos
    GROUP BY FK_LOJA, DATA
)
SELECT
    -- Chaves de Granularidade Diária
    dv.DATA,
    dv.LOJA,
    il.FK_LOJA,
    dv.FK_SECAO,

    -- Variável Target
    dv.VALOR AS TARGET_VENDAS_DIARIAS_EUR,

    -- Features Temporais
    dv.MES,
    dv.ANO,
    dv.SEMANA_DO_ANO,
    dv.DIA_SEMANA,

    -- Features: Dados de Vendas
    dv.`SKUS_+`,
    dv.`SKUS_-`,
    dv.`VAR_PREÇO_+`,
    dv.`VAR_PREÇO_-`,

    -- Features: Informação da Loja
    il.CIDADE,
    il.REGIAO,
    il.`PRODUTIVIDADE/HORA`,
    il.N_COLABORADORES,
    il.SKUS AS TOTAL_SKUS_LOJA,
    il.ABERTURA,
    il.FECHO,
    il.CAIXAS_TRADICIONAIS,
    il.SELF_CHECKOUTS,

    -- Features: Feriados
    f.DESCRICAO AS FERIADO,
    CASE WHEN f.DESCRICAO = 'Ano Novo' THEN 1 ELSE 0 END AS ANO_NOVO,
    CASE WHEN f.DESCRICAO = 'Dia do Trabalhador' THEN 1 ELSE 0 END AS DIA_DO_TRABALHADOR,
    CASE WHEN f.DESCRICAO = 'Natal' THEN 1 ELSE 0 END AS NATAL,
    CASE WHEN f.DESCRICAO LIKE '%Páscoa%' OR f.DESCRICAO LIKE '%Pascoa%' THEN 1 ELSE 0 END AS PASCOA,
    CASE WHEN f.FERIADO_FIXO = 'S' THEN 1 ELSE 0 END AS FERIADO_FIXO,
    COALESCE(f.LOJA_ABERTA, 1) AS LOJA_ABERTA,

    -- Features: Eventos
    CASE WHEN e.EVENTOS IS NOT NULL THEN 1 ELSE 0 END AS IS_EVENT

FROM retailcast_solo.retailcast_solo_silver.rc_dados_vendas_silver dv

-- JOIN 1: Dimensão da Loja usando LOJA
LEFT JOIN retailcast_solo.retailcast_solo_silver.rc_info_loja_silver il
    ON dv.LOJA = il.LOJA

-- JOIN 2: Feriados pontuais alinhados com a FK_LOJA e DATA
LEFT JOIN retailcast_solo.retailcast_solo_silver.rc_feriados_silver f
    ON il.FK_LOJA = f.FK_LOJA
    AND TO_DATE(dv.DATA) = TO_DATE(f.DATA)

-- JOIN 3: Eventos diários agregados alinhados com a FK_LOJA e DATA
LEFT JOIN Eventos_Diarios e
    ON il.FK_LOJA = e.FK_LOJA
    AND TO_DATE(dv.DATA) = e.DATA;

num_affected_rows,num_inserted_rows
